# Pit Stop Submission Blender: 0.95449 Anchor + External Blend Research

We found several new public solutions from other participants. One notebook uses a very direct idea: blend two strong submissions with a simple `50/50` average. Another notebook simply copies a ready-made `9446.csv` submission, which has a very different, rank-like distribution. Together with the new `0.95449.csv`, these files give us a new stronger base and a new diversity source.

This notebook now treats `0.95449` as the main anchor. The goal is to test a small, readable set of blend methods around it: direct probability blends with close high-scoring submissions, rank-remap blends using the unusual `0.95446` source, HB10-style row-wise h-blends, and a few selective corrections. The output remains compact: `outputs/max` contains the first files to submit, `outputs/pro` contains secondary diagnostics, and `outputs/report.csv` explains every saved file.


In [ ]:
"""Import libraries, configure paths, and define compact display helpers."""

from pathlib import Path
from html import escape
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import seaborn as sns
except ModuleNotFoundError:
    sns = None
from IPython.display import display, HTML

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")
if sns is not None:
    sns.set_theme(style="whitegrid", context="notebook")
else:
    plt.style.use("ggplot")

ID_COL = "id"
TARGET_COL = "PitNextLap"
CLIP_LOW = 1e-7
CLIP_HIGH = 1 - 1e-7

DATASET_CANDIDATES = [
    Path("/kaggle/input/pitstop-blend-inputs"),
    Path("/kaggle/input/pit-stop-blend-inputs"),
    Path("/kaggle/input/blend-dataset"),
    Path("/kaggle/input") / "blend_dataset",
    Path("competitions/predicting-pit-stop/blend_dataset"),
    Path("PredictingPitStop/blend_dataset"),
    Path("blend_dataset"),
]

OUTPUT_ROOT = Path("outputs")
MAX_ROOT = OUTPUT_ROOT / "max"
PRO_ROOT = OUTPUT_ROOT / "pro"
DIAGNOSTIC_ROOT = OUTPUT_ROOT / "diagnostics"
REPORT_PATH = OUTPUT_ROOT / "report.csv"

for folder in [MAX_ROOT, PRO_ROOT, DIAGNOSTIC_ROOT]:
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True, exist_ok=True)
if REPORT_PATH.exists():
    REPORT_PATH.unlink()


def find_dataset_dir():
    for path in DATASET_CANDIDATES:
        if (path / "public").exists() and (path / "ours").exists():
            return path
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for path in sorted(kaggle_input.rglob("*")):
            if path.is_dir() and (path / "public").exists() and (path / "ours").exists():
                return path
    raise FileNotFoundError("Could not find a dataset folder containing public/ and ours/ subfolders.")


def show_title(title, subtitle=None):
    subtitle_html = f'<div style="color:#6b7280;font-size:13px;margin-top:4px;">{escape(str(subtitle))}</div>' if subtitle else ""
    display(HTML(f"""
    <div style="margin:20px 0 12px 0;padding-bottom:9px;border-bottom:1px solid #e5e7eb;">
      <div style="font-size:20px;font-weight:750;color:#111827;">{escape(str(title))}</div>
      {subtitle_html}
    </div>
    """))


def show_cards(metrics, columns=4):
    width = max(1, int(100 / columns))
    cards = []
    for label, value in metrics.items():
        if isinstance(value, float):
            value = f"{value:.6f}".rstrip("0").rstrip(".")
        elif isinstance(value, int):
            value = f"{value:,}"
        cards.append(f"""
        <div style="box-sizing:border-box;width:{width}%;padding:6px;">
          <div style="border:1px solid #e5e7eb;border-radius:8px;padding:12px;background:#fff;">
            <div style="font-size:12px;color:#6b7280;text-transform:uppercase;letter-spacing:.03em;">{escape(str(label))}</div>
            <div style="font-size:21px;font-weight:750;color:#111827;margin-top:5px;">{escape(str(value))}</div>
          </div>
        </div>
        """)
    display(HTML(f'<div style="display:flex;flex-wrap:wrap;margin:0 -6px 14px -6px;">{"".join(cards)}</div>'))


def show_table(title, df, max_rows=12, precision=6):
    shown = df.head(max_rows).copy()
    display(HTML(f'<div style="font-size:16px;font-weight:700;color:#111827;margin:14px 0 6px;">{escape(title)}</div>'))
    styler = shown.style.format(precision=precision).set_table_styles([
        {"selector":"th", "props":[("background", "#f9fafb"), ("color", "#374151"), ("font-weight", "700"), ("border-bottom", "1px solid #e5e7eb")]},
        {"selector":"td", "props":[("border-bottom", "1px solid #f3f4f6"), ("font-size", "13px")]},
        {"selector":"table", "props":[("border-collapse", "collapse"), ("width", "100%")]},
    ])
    display(styler)
    if len(df) > max_rows:
        display(HTML(f'<div style="color:#6b7280;font-size:12px;margin-top:4px;">Showing {max_rows} of {len(df)} rows.</div>'))


def corr(a, b):
    return float(np.corrcoef(np.asarray(a), np.asarray(b))[0, 1])


def mean_abs_delta(a, b):
    return float(np.abs(np.asarray(a) - np.asarray(b)).mean())


def clip_pred(pred):
    return np.clip(np.asarray(pred, dtype=float), CLIP_LOW, CLIP_HIGH)


## 1. Load Blend Dataset

The dataset contains normal probability submissions in `public/super` and a separate rank-like source in `public/rank_diverse`. Keeping those groups separate matters because `9446.csv` has a different scale and should be used through rank-remapping rather than direct probability averaging.


In [ ]:
"""Load all valid submission files from the structured blend dataset."""

dataset_dir = find_dataset_dir()
predictions = {}
records = []
base_ids = None


def load_submission(path, source):
    global base_ids
    df = pd.read_csv(path)
    if ID_COL not in df.columns:
        return None
    target_candidates = [col for col in df.columns if col != ID_COL]
    if TARGET_COL in df.columns:
        target_col = TARGET_COL
    elif len(target_candidates) == 1:
        target_col = target_candidates[0]
    else:
        return None

    df = df[[ID_COL, target_col]].rename(columns={target_col: TARGET_COL})
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    if df[TARGET_COL].isna().any() or df[ID_COL].duplicated().any():
        return None

    if base_ids is None:
        base_ids = df[ID_COL].copy()
    elif not base_ids.equals(df[ID_COL]):
        return None

    group = path.parent.name
    safe_stem = path.stem.replace(".", "_").replace("-", "_")
    name = f"{source}_{group}_{safe_stem}"
    pred = clip_pred(df[TARGET_COL].to_numpy(dtype=float))
    predictions[name] = pred

    public_score = np.nan
    if source == "public":
        try:
            public_score = float(path.stem.split("_")[0])
        except ValueError:
            public_score = np.nan

    return {"name": name, "source": source, "group": group, "public_score": public_score, "rows": len(df), "mean": pred.mean(), "std": pred.std(), "file": str(path.relative_to(dataset_dir))}

for source in ["public", "ours"]:
    folder = dataset_dir / source
    if folder.exists():
        for path in sorted(folder.rglob("*.csv")):
            row = load_submission(path, source)
            if row is not None:
                records.append(row)

if not records:
    raise FileNotFoundError("No valid submissions were loaded from the structured dataset.")

input_summary = pd.DataFrame(records).sort_values(["source", "group", "public_score", "name"], ascending=[True, True, False, True]).reset_index(drop=True)
input_summary.to_csv(DIAGNOSTIC_ROOT / "inputs.csv", index=False)

show_title("Loaded blend dataset", f"Dataset folder: {dataset_dir}")
show_cards({"input files": len(input_summary), "public files": int(input_summary["source"].eq("public").sum()), "own files": int(input_summary["source"].eq("ours").sum()), "rows": len(base_ids)})
show_table("Structured inputs", input_summary[["name", "source", "group", "public_score", "mean", "std", "file"]], max_rows=32)


## 2. Anchor And Source Diagnostics

`0.95449` becomes the main anchor. We compare it with the previous anchors and inspect the unusual `0.95446` rank-like source separately. The goal is to identify which files are safe for direct probability blends and which should only influence ranking.


In [ ]:
"""Select the new anchor and compare probability and rank-diverse support sources."""

super_meta = input_summary[input_summary["group"].eq("super")].sort_values("public_score", ascending=False).copy()
rank_meta = input_summary[input_summary["group"].eq("rank_diverse")].sort_values("public_score", ascending=False).copy()


def names_by_score(meta, score):
    return meta[np.isclose(meta["public_score"], score, atol=1e-8)]["name"].tolist()

s49_names = names_by_score(super_meta, 0.95449)
if not s49_names:
    raise RuntimeError("The 0.95449 submission is required in public/super.")
s49_name = s49_names[0]
s49_pred = predictions[s49_name]

s37_pred = predictions[names_by_score(super_meta, 0.95437)[0]]
s35_pred = predictions[names_by_score(super_meta, 0.95435)[0]]
s31_pred = clip_pred(np.vstack([predictions[name] for name in names_by_score(super_meta, 0.95431)]).mean(axis=0))
s19_pred = predictions[names_by_score(super_meta, 0.95419)[0]]
s18_pred = predictions[names_by_score(super_meta, 0.95418)[0]]
s11_pred = clip_pred(np.vstack([predictions[name] for name in names_by_score(super_meta, 0.95411)]).mean(axis=0))

if rank_meta.empty:
    raise RuntimeError("The rank_diverse group with 0.95446 is required.")
s46_name = rank_meta.iloc[0]["name"]
s46_pred = predictions[s46_name]

source_signals = {"s49": s49_pred, "s46_rank": s46_pred, "s37": s37_pred, "s35": s35_pred, "s31": s31_pred, "s19": s19_pred, "s18": s18_pred, "s11": s11_pred}
source_summary = pd.DataFrame([
    {"signal": name, "mean": pred.mean(), "std": pred.std(), "corr_to_s49": corr(pred, s49_pred), "delta_to_s49": mean_abs_delta(pred, s49_pred)}
    for name, pred in source_signals.items()
]).sort_values("delta_to_s49").reset_index(drop=True)
source_summary.to_csv(DIAGNOSTIC_ROOT / "source_diagnostics.csv", index=False)

show_title("Anchor and source diagnostics", "0.95449 is the new anchor; 0.95446 is a rank-like source")
show_cards({"anchor": "0.95449", "s37 delta": mean_abs_delta(s37_pred, s49_pred), "s46 corr": corr(s46_pred, s49_pred), "s46 mean": s46_pred.mean()})
show_table("Source summary", source_summary, max_rows=len(source_summary))

corr_df = pd.DataFrame(source_signals).corr()
fig, ax = plt.subplots(figsize=(8, 5.5))
if sns is not None:
    sns.heatmap(corr_df, cmap="viridis", annot=True, fmt=".4f", cbar=False, ax=ax)
else:
    im = ax.imshow(corr_df.values, cmap="viridis")
    ax.set_xticks(range(len(corr_df.columns)), corr_df.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(corr_df.index)), corr_df.index)
    fig.colorbar(im, ax=ax)
ax.set_title("Source correlation")
plt.tight_layout()
plt.show()


## 3. Build Support Signals

This block prepares reusable support predictions. The probability consensus is used for ordinary blends, while the rank-like `s46` is kept separate and is not directly averaged as a probability.


In [ ]:
"""Build probability support and consensus signals used by later methods."""


def group_names(group):
    return input_summary[input_summary["group"].eq(group)]["name"].tolist()


def mean_predictions(names):
    if not names:
        raise RuntimeError("Cannot average an empty list of predictions.")
    return clip_pred(np.vstack([predictions[name] for name in names]).mean(axis=0))

public_core = input_summary[input_summary["group"].eq("core")].sort_values("public_score", ascending=False)["name"].head(6).tolist()
public_diverse = group_names("diverse")
top_external = group_names("top_external")
core_pred = mean_predictions(public_core)
diverse_pred = predictions[public_diverse[0]]
b10_pred = clip_pred(0.950 * core_pred + 0.050 * diverse_pred)
tx_pred = mean_predictions(top_external)
tb_pred = clip_pred(0.900 * tx_pred + 0.100 * b10_pred)

prob_consensus = clip_pred(0.45 * s37_pred + 0.30 * s35_pred + 0.25 * s31_pred)
broad_consensus = clip_pred(0.50 * s37_pred + 0.20 * s35_pred + 0.15 * s31_pred + 0.15 * s19_pred)

support_summary = pd.DataFrame([
    {"signal": name, "mean": pred.mean(), "std": pred.std(), "corr_to_s49": corr(pred, s49_pred), "delta_to_s49": mean_abs_delta(pred, s49_pred)}
    for name, pred in {"s49": s49_pred, "prob_consensus": prob_consensus, "broad_consensus": broad_consensus, "tb": tb_pred, "s46_rank": s46_pred}.items()
]).sort_values("delta_to_s49").reset_index(drop=True)
support_summary.to_csv(DIAGNOSTIC_ROOT / "support_signals.csv", index=False)

show_title("Support signals", "Probability consensus and rank-like source are kept separate")
show_table("Support summary", support_summary, max_rows=len(support_summary))


## 4. Method 1: Direct Probability Blends

The direct-blend notebook showed that a simple average of two strong probability submissions can work. Here we adapt that idea conservatively: `s49` stays dominant, and close high-scoring supports only add a small correction.


In [ ]:
"""Create direct probability-blend candidates around s49."""

prob_candidates = {
    "d37_10": {"pred": 0.900 * s49_pred + 0.100 * s37_pred, "formula": "0.900*s49 + 0.100*s37", "tier": "max", "priority": 2, "reason": "Main direct blend with the previous 0.95437 anchor."},
    "d37_05": {"pred": 0.950 * s49_pred + 0.050 * s37_pred, "formula": "0.950*s49 + 0.050*s37", "tier": "pro", "priority": 7, "reason": "Smaller direct correction from s37."},
    "d31_05": {"pred": 0.950 * s49_pred + 0.050 * s31_pred, "formula": "0.950*s49 + 0.050*s31", "tier": "pro", "priority": 8, "reason": "Direct blend with the more diverse 0.95431 support."},
}

prob_summary = pd.DataFrame([
    {"candidate": key, "tier": spec["tier"], "formula": spec["formula"], "corr_to_s49": corr(spec["pred"], s49_pred), "delta_to_s49": mean_abs_delta(spec["pred"], s49_pred), "reason": spec["reason"]}
    for key, spec in prob_candidates.items()
]).sort_values("delta_to_s49")
prob_summary.to_csv(DIAGNOSTIC_ROOT / "probability_candidates.csv", index=False)

show_title("Direct probability blends", "Small corrections from normal probability submissions")
show_table("Probability candidates", prob_summary, max_rows=len(prob_summary))


## 5. Method 2: Rank-Remap Blends

The `0.95446` file has a very different distribution, so it should not be mixed as probability. Instead, we blend ranks and map the result back to the `s49` probability distribution. This tests whether `0.95446` carries useful ordering information.


In [ ]:
"""Create rank-remap candidates using both normal and rank-like support sources."""


def normalized_rank(values):
    order = np.argsort(values, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.linspace(0.0, 1.0, len(values))
    return ranks


def rank_blend(anchor, support, support_weight):
    anchor_rank = normalized_rank(anchor)
    support_rank = normalized_rank(support)
    blended_rank = (1 - support_weight) * anchor_rank + support_weight * support_rank
    order = np.argsort(blended_rank, kind="mergesort")
    sorted_anchor_values = np.sort(anchor)
    out = np.empty_like(anchor, dtype=float)
    out[order] = sorted_anchor_values
    return clip_pred(out)

rank_candidates = {
    "r46_02": {"pred": rank_blend(s49_pred, s46_pred, 0.020), "formula": "rank 0.980*s49 + 0.020*s46", "tier": "max", "priority": 3, "reason": "Main rank-remap test using the 0.95446 rank-like source."},
    "r46_05": {"pred": rank_blend(s49_pred, s46_pred, 0.050), "formula": "rank 0.950*s49 + 0.050*s46", "tier": "max", "priority": 4, "reason": "Stronger rank-remap test with the rank-like source."},
    "r37_05": {"pred": rank_blend(s49_pred, s37_pred, 0.050), "formula": "rank 0.950*s49 + 0.050*s37", "tier": "pro", "priority": 9, "reason": "Rank test with the previous probability anchor."},
}

rank_summary = pd.DataFrame([
    {"candidate": key, "tier": spec["tier"], "formula": spec["formula"], "corr_to_s49": corr(spec["pred"], s49_pred), "delta_to_s49": mean_abs_delta(spec["pred"], s49_pred), "reason": spec["reason"]}
    for key, spec in rank_candidates.items()
]).sort_values("delta_to_s49")
rank_summary.to_csv(DIAGNOSTIC_ROOT / "rank_candidates.csv", index=False)

show_title("Rank-remap blends", "Use s46 as ordering signal, not probability signal")
show_table("Rank candidates", rank_summary, max_rows=len(rank_summary))


## 6. Method 3: HB10-Style H-Blend

The HB10-style method adjusts source weights per row depending on row-wise ordering. Here it is centered on `s49`, with `s37`, `s35`, and `s31` as normal probability supports. The rank-like `s46` stays out of this probability h-blend.


In [ ]:
"""Create HB10-style row-wise h-blend candidates with s49 as the dominant source."""


def h_blend_matrix(named_preds, base_weights, subwts, desc_weight=0.70):
    names = list(named_preds)
    matrix = np.vstack([named_preds[name] for name in names]).T
    base = np.asarray([base_weights[name] for name in names], dtype=float)
    corr_w = np.asarray(subwts, dtype=float)
    if len(corr_w) != len(names):
        raise ValueError("subwts length must match number of h-blend submissions.")

    def side(reverse):
        order = np.argsort(-matrix if reverse else matrix, axis=1)
        out = np.zeros(matrix.shape[0], dtype=float)
        for rank_idx in range(matrix.shape[1]):
            col_idx = order[:, rank_idx]
            weights = base[col_idx] + corr_w[rank_idx]
            out += matrix[np.arange(matrix.shape[0]), col_idx] * weights
        return out

    desc = side(True)
    asc = side(False)
    return clip_pred(desc_weight * desc + (1 - desc_weight) * asc)

h_sources = {"s49": s49_pred, "s37": s37_pred, "s35": s35_pred, "s31": s31_pred}
hb49_pred = h_blend_matrix(h_sources, {"s49": 0.58, "s37": 0.18, "s35": 0.14, "s31": 0.10}, [-0.035, 0.005, 0.012, 0.018], 0.70)
hc49_pred = clip_pred(0.850 * s49_pred + 0.150 * hb49_pred)
hbold_pred = h_blend_matrix(h_sources, {"s49": 0.50, "s37": 0.22, "s35": 0.16, "s31": 0.12}, [-0.055, 0.010, 0.018, 0.027], 0.70)

hblend_candidates = {
    "hb49": {"pred": hb49_pred, "formula": "HB10-style h-blend: s49/s37/s35/s31", "tier": "max", "priority": 5, "reason": "Main row-wise h-blend centered on the new anchor."},
    "hc49": {"pred": hc49_pred, "formula": "0.850*s49 + 0.150*hb49", "tier": "pro", "priority": 10, "reason": "Conservative h-blend correction pulled toward the anchor."},
    "hbold": {"pred": hbold_pred, "formula": "bolder HB10-style h-blend: s49/s37/s35/s31", "tier": "pro", "priority": 11, "reason": "More aggressive row-wise h-blend diagnostic."},
}

hblend_summary = pd.DataFrame([
    {"candidate": key, "tier": spec["tier"], "formula": spec["formula"], "corr_to_s49": corr(spec["pred"], s49_pred), "delta_to_s49": mean_abs_delta(spec["pred"], s49_pred), "reason": spec["reason"]}
    for key, spec in hblend_candidates.items()
]).sort_values("delta_to_s49")
hblend_summary.to_csv(DIAGNOSTIC_ROOT / "hblend_candidates.csv", index=False)

show_title("HB10-style h-blends", "Row-wise probability blending around s49")
show_table("H-blend candidates", hblend_summary, max_rows=len(hblend_summary))


## 7. Method 4: Selective Corrections

Selective corrections change only a small set of rows where the anchor and support disagree most. This is useful if a source helps on a small number of hard rows but hurts when blended globally.


In [ ]:
"""Create selective correction candidates on the largest anchor/support disagreements."""


def selective_blend(anchor, support, fraction, support_weight, extreme_only=False):
    anchor = np.asarray(anchor, dtype=float)
    support = np.asarray(support, dtype=float)
    delta = support - anchor
    eligible = np.ones_like(delta, dtype=bool)
    if extreme_only:
        ranks = normalized_rank(anchor)
        eligible = eligible & ((ranks <= 0.10) | (ranks >= 0.90))
    scores = np.where(eligible, np.abs(delta), -np.inf)
    n_select = max(1, int(round(len(anchor) * fraction)))
    selected = np.argpartition(scores, -n_select)[-n_select:]
    selected = selected[np.isfinite(scores[selected])]
    out = anchor.copy()
    out[selected] = (1 - support_weight) * anchor[selected] + support_weight * support[selected]
    return clip_pred(out), len(selected)

c37_pred, c37_rows = selective_blend(s49_pred, s37_pred, fraction=0.020, support_weight=0.100)
cc_pred, cc_rows = selective_blend(s49_pred, broad_consensus, fraction=0.020, support_weight=0.100)
ex37_pred, ex37_rows = selective_blend(s49_pred, s37_pred, fraction=0.020, support_weight=0.100, extreme_only=True)

selective_candidates = {
    "c37": {"pred": c37_pred, "formula": "top 2% |s37-s49|: 0.900*s49 + 0.100*s37", "tier": "pro", "priority": 12, "changed_rows": c37_rows, "reason": "Selective correction from the previous anchor."},
    "cc": {"pred": cc_pred, "formula": "top 2% |consensus-s49|: 0.900*s49 + 0.100*consensus", "tier": "pro", "priority": 13, "changed_rows": cc_rows, "reason": "Selective correction from the broader probability consensus."},
    "ex37": {"pred": ex37_pred, "formula": "extreme top/bottom 10%, top 2% |s37-s49| correction", "tier": "pro", "priority": 14, "changed_rows": ex37_rows, "reason": "Extreme-rank correction for AUC-sensitive rows."},
}

selective_summary = pd.DataFrame([
    {"candidate": key, "tier": spec["tier"], "formula": spec["formula"], "changed_rows": spec["changed_rows"], "corr_to_s49": corr(spec["pred"], s49_pred), "delta_to_s49": mean_abs_delta(spec["pred"], s49_pred), "reason": spec["reason"]}
    for key, spec in selective_candidates.items()
]).sort_values("delta_to_s49")
selective_summary.to_csv(DIAGNOSTIC_ROOT / "selective_candidates.csv", index=False)

show_title("Selective corrections", "Small row-level changes around the anchor")
show_table("Selective candidates", selective_summary, max_rows=len(selective_summary))


## 8. Save Final Outputs

The saved files are separated by priority. `max` contains the most important submissions to test first. `pro` contains secondary diagnostics that should guide the next narrow iteration if one method family works.


In [ ]:
"""Save selected candidates to outputs/max and outputs/pro, then build one report file."""


def save_submission(path, pred):
    pred = clip_pred(pred)
    pd.DataFrame({ID_COL: base_ids.values, TARGET_COL: pred}).to_csv(path, index=False)

all_candidates = {"s49": {"pred": s49_pred, "method": "anchor", "tier": "max", "priority": 1, "formula": "0.95449 raw anchor", "changed_rows": 0, "reason": "Verify the new strongest anchor unchanged."}}
for key, spec in prob_candidates.items():
    all_candidates[key] = {"method": "direct_probability", "changed_rows": len(s49_pred), **spec}
for key, spec in rank_candidates.items():
    all_candidates[key] = {"method": "rank_remap", "changed_rows": len(s49_pred), **spec}
for key, spec in hblend_candidates.items():
    all_candidates[key] = {"method": "hblend", "changed_rows": len(s49_pred), **spec}
for key, spec in selective_candidates.items():
    all_candidates[key] = {"method": "selective", **spec}

rows = []
for key, spec in sorted(all_candidates.items(), key=lambda item: item[1]["priority"]):
    pred = clip_pred(spec["pred"])
    folder = MAX_ROOT if spec["tier"] == "max" else PRO_ROOT
    file_path = folder / f"{key}.csv"
    save_submission(file_path, pred)
    rows.append({"priority": spec["priority"], "tier": spec["tier"], "candidate": key, "method": spec["method"], "file": str(file_path.relative_to(OUTPUT_ROOT)), "formula": spec["formula"], "mean": pred.mean(), "std": pred.std(), "corr_to_s49": corr(pred, s49_pred), "delta_to_s49": mean_abs_delta(pred, s49_pred), "changed_rows": spec["changed_rows"], "reason": spec["reason"]})

report = pd.DataFrame(rows).sort_values("priority").reset_index(drop=True)
report.to_csv(REPORT_PATH, index=False)

show_title("Saved output files", "Upload max first; use pro for method diagnostics")
show_cards({"max files": int(report["tier"].eq("max").sum()), "pro files": int(report["tier"].eq("pro").sum()), "total files": len(report), "report": "outputs/report.csv"})
show_table("Submit order", report[["priority", "tier", "candidate", "method", "file", "formula", "delta_to_s49", "changed_rows", "reason"]], max_rows=len(report))


## 9. Candidate Movement

These plots make the output easier to audit. If a max candidate improves the leaderboard, the next iteration should tune only that method family instead of widening the search randomly.


In [ ]:
"""Visualize candidate movement and correction shape relative to the new anchor."""

plot_df = report[report["candidate"].ne("s49")].copy()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
if sns is not None:
    sns.barplot(data=plot_df, y="candidate", x="delta_to_s49", hue="method", dodge=False, ax=axes[0])
    sns.barplot(data=plot_df, y="candidate", x="changed_rows", hue="method", dodge=False, ax=axes[1])
    for ax in axes:
        legend = ax.get_legend()
        if legend is not None:
            legend.remove()
else:
    axes[0].barh(plot_df["candidate"], plot_df["delta_to_s49"])
    axes[1].barh(plot_df["candidate"], plot_df["changed_rows"])
axes[0].set_title("Movement from s49")
axes[0].set_xlabel("mean absolute delta")
axes[1].set_title("Rows changed")
axes[1].set_xlabel("rows")
for ax in axes:
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7.5, 4.3))
for candidate in ["d37_10", "r46_02", "hb49"]:
    row = report[report["candidate"].eq(candidate)].iloc[0]
    pred = pd.read_csv(OUTPUT_ROOT / row["file"])[TARGET_COL].to_numpy(dtype=float)
    ax.hist(pred - s49_pred, bins=80, alpha=0.45, label=candidate)
ax.set_title("Main candidate corrections relative to s49")
ax.set_xlabel("prediction delta")
ax.set_ylabel("rows")
ax.legend()
plt.tight_layout()
plt.show()

show_title("Final recommendation", "The max files test the main hypotheses")
show_table("Max candidates", report[report["tier"].eq("max")][["priority", "candidate", "method", "file", "reason"]], max_rows=10)
show_table("Pro candidates", report[report["tier"].eq("pro")][["priority", "candidate", "method", "file", "reason"]], max_rows=20)


## Final Notes

The first files to try are `s49`, `d37_10`, `r46_02`, `r46_05`, and `hb49`. They test different hypotheses: raw anchor strength, direct probability blending, rank-remapping from the unusual `0.95446` source, and row-wise h-blending.

If none of the max files improves `0.95449`, the current external source set may again be near its local limit. If one method improves, the next notebook should narrow the search around that method only.
